In [ ]:
## Base library Installation
# Install Baselines for model comparison
!uv pip install catboost xgboost

# Install the datasets library for loading example data
!uv pip install datasets

# Install rich for better and more readable printing
!uv pip install rich


## TabPFN Installation optimized for Google Colab
# Install the TabPFN Client library
!uv pip install tabpfn-client

# Install tabpfn from source
# Clone the repository: shallow for speedup
!git clone --depth 1 https://github.com/PriorLabs/tabpfn

# Speeding up installation in this notebook:
# Remove torch dependency as it is already installed on colab (do not run this in your local setup)
!sed -i "/torch/d" tabpfn/pyproject.toml

# Step 3: Install using the correct directory name 'tabpfn'
!uv pip install -e "tabpfn"

# Speeding up installation in this notebook:
# Remove torch dependency as it is already installed on colab (do not run this in your local setup)
!sed -i "/torch/d" tabpfn-extensions/pyproject.toml

!uv pip install -e tabpfn-extensions[all]

Using Python 3.12.12 environment at: /usr
Resolved 21 packages in 336ms
Prepared 1 package in 1.36s
Installed 1 package in 4ms
 + catboost==1.2.10
Using Python 3.12.12 environment at: /usr
Audited 1 package in 108ms
Using Python 3.12.12 environment at: /usr
Audited 1 package in 97ms
Using Python 3.12.12 environment at: /usr
Resolved 33 packages in 220ms
Prepared 5 packages in 86ms
Uninstalled 1 package in 2ms
Installed 5 packages in 5ms
 + backoff==2.2.1
 + password-strength==0.0.3.post2
 + sseclient-py==1.8.0
 + tabpfn-client==0.2.8
 - tqdm==4.67.3
 + tqdm==4.67.1
Cloning into 'tabpfn'...
remote: Enumerating objects: 254, done.
remote: Counting objects: 100% (254/254), done.
remote: Compressing objects: 100% (227/227), done.
remote: Total 254 (delta 32), reused 109 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (254/254), 2.01 MiB | 6.41 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Using Python 3.12.12 environment at: /usr
Resolved 51 packages in 333ms
Prepared 5 pac

In [ ]:
pip install --upgrade tabpfn

In [ ]:
import tabpfn
print("TabPFN imported successfully!")

TabPFN imported successfully!


In [ ]:
# Standard Library Imports

# TabPFN and Extensions

try:
    from tabpfn import TabPFNClassifier, TabPFNRegressor
    from tabpfn_extensions.post_hoc_ensembles.sklearn_interface import (
        AutoTabPFNClassifier,
    )
except ImportError:
    raise ImportError(
        "Warning: Could not import TabPFN / TabPFN extensions. Please run installation above and restart the session afterwards (Runtime > Restart Session)."
    )

# Data Science & Visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# Other ML Models
from catboost import CatBoostClassifier, CatBoostRegressor

# Notebook UI/Display
from IPython.display import Markdown, display
from rich.console import Console
from rich.panel import Panel
from rich.prompt import Prompt
from rich.rule import Rule
from sklearn.compose import make_column_selector, make_column_transformer

# Scikit-Learn: Data & Preprocessing
from sklearn.datasets import fetch_openml, load_breast_cancer

# Scikit-Learn: Models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from xgboost import XGBClassifier, XGBRegressor

# This transformer will be used to handle categorical features for the baseline models
column_transformer = make_column_transformer(
    (
        OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        make_column_selector(dtype_include=["object", "category"]),
    ),
    remainder="passthrough",
)

In [ ]:
from sklearn.utils import shuffle
from pathlib import Path

In [ ]:
def split_data(X, Y, p = 0.8, seed = 123456):
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size = p, random_state = seed)
    return X_train, X_test, Y_train, Y_test

In [ ]:
def performance_eval(Y_pred, Y_test):
    Y_pred = np.array(Y_pred)
    Y_test = np.array(Y_test)
    MAE = np.mean(np.abs(Y_pred - Y_test))
    RMSE = np.sqrt(np.mean((Y_pred - Y_test)**2))
    MAPE = np.mean(np.abs(Y_test - Y_pred) / ((np.abs(Y_test) + np.abs(Y_pred)) / 2)) * 100

    #accuracy = np.mean(Y_pred == Y_test) * 100

    return MAE, RMSE, MAPE

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Base

ICL4000

In [ ]:
# Set up the data directory path
DATA_DIR = Path("/content/drive/My Drive/tabPFN_update_data_March/linear_exp_all_positive2")

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_base_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_base_2.csv")

tabpfn-v2.5-regressor-v2.5_default.ckpt:   0%|          | 0.00/40.8M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

In [ ]:
df

,csv,mae,rmse,mape
0,/content/drive/My Drive/tabPFN_update_data_Mar...,0.081556,0.102841,1.567982
1,/content/drive/My Drive/tabPFN_update_data_Mar...,0.082013,0.102838,1.622304
2,/content/drive/My Drive/tabPFN_update_data_Mar...,0.081595,0.105241,1.566043
3,/content/drive/My Drive/tabPFN_update_data_Mar...,0.079988,0.100885,1.548439
4,/content/drive/My Drive/tabPFN_update_data_Mar...,0.082708,0.105112,1.600430
...,...,...,...,...
95,/content/drive/My Drive/tabPFN_update_data_Mar...,0.086013,0.110557,1.611169
96,/content/drive/My Drive/tabPFN_update_data_Mar...,0.081997,0.107874,1.567558
97,/content/drive/My Drive/tabPFN_update_data_Mar...,0.083832,0.105916,1.608927
98,/content/drive/My Drive/tabPFN_update_data_Mar...,0.085973,0.108450,1.672668


ICL10

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:10,:]
    Y_train = Y_train[:10]

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_base_fewshot10_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_base_fewshot10_2.csv")

ICL20

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:20,:]
    Y_train = Y_train[:20]

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_base_fewshot20_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_base_fewshot20_2.csv")

ICL500

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:500,:]
    Y_train = Y_train[:500]

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_base_fewshot500_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_base_fewshot500_2.csv")

## Row Order

ICL4000

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)

    X_train = X_train.iloc[::-1]
    Y_train = Y_train.iloc[::-1]

    model_row = TabPFNRegressor(random_state=123456)
    model_row.fit(X_train, Y_train)
    y_pred = model_row.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_rowOrder_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_rowOrder_2.csv")

In [ ]:
df

,csv,mae,rmse,mape
0,/content/drive/My Drive/tabPFN_update_data_Mar...,0.081566,0.102846,1.568101
1,/content/drive/My Drive/tabPFN_update_data_Mar...,0.081984,0.102805,1.621691
2,/content/drive/My Drive/tabPFN_update_data_Mar...,0.081611,0.105233,1.566133
3,/content/drive/My Drive/tabPFN_update_data_Mar...,0.080001,0.100904,1.548582
4,/content/drive/My Drive/tabPFN_update_data_Mar...,0.082691,0.105088,1.600143
...,...,...,...,...
95,/content/drive/My Drive/tabPFN_update_data_Mar...,0.086015,0.110552,1.611117
96,/content/drive/My Drive/tabPFN_update_data_Mar...,0.081986,0.107843,1.567571
97,/content/drive/My Drive/tabPFN_update_data_Mar...,0.083795,0.105808,1.608830
98,/content/drive/My Drive/tabPFN_update_data_Mar...,0.085976,0.108456,1.672762


ICL10

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:10,:]
    Y_train = Y_train[:10]
    X_train = X_train.iloc[::-1]
    Y_train = Y_train.iloc[::-1]

    model_row = TabPFNRegressor(random_state=123456)
    model_row.fit(X_train, Y_train)
    y_pred = model_row.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_rowOrder_fewshot10_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_rowOrder_fewshot10_2.csv")

ICL20

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:20,:]
    Y_train = Y_train[:20]
    X_train = X_train.iloc[::-1]
    Y_train = Y_train.iloc[::-1]

    model_row = TabPFNRegressor(random_state=123456)
    model_row.fit(X_train, Y_train)
    y_pred = model_row.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_rowOrder_fewshot20_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_rowOrder_fewshot20_2.csv")

ICL500

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:500,:]
    Y_train = Y_train[:500]
    X_train = X_train.iloc[::-1]
    Y_train = Y_train.iloc[::-1]

    model_row = TabPFNRegressor(random_state=123456)
    model_row.fit(X_train, Y_train)
    y_pred = model_row.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_rowOrder_fewshot500_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_rowOrder_fewshot500_2.csv")

# Num_Digits

ICL4000

In [ ]:
DATA_DIR = Path("/content/drive/My Drive/tabPFN_update_data_March/linear_exp_all_positive2_10_digits")

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_10_digits_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_10_digits_2.csv")

ICL10

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:10,:]
    Y_train = Y_train[:10]

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_10_digits_fewshot10_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_10_digits_fewshot10_2.csv")

ICL20

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:20,:]
    Y_train = Y_train[:20]

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_10_digits_fewshot20_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_10_digits_fewshot20_2.csv")

ICL500

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:500,:]
    Y_train = Y_train[:500]

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_10_digits_fewshot500_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_10_digits_fewshot500_2.csv")

## Column_Order

ICL4000

In [ ]:
DATA_DIR = Path("/content/drive/My Drive/tabPFN_update_data_March/linear_exp_all_positive2_shuffle")

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_shuffle_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_shuffle_2.csv")

ICL10

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:10,:]
    Y_train = Y_train[:10]

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_shuffle_fewshot10_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_shuffle_fewshot10_2.csv")

ICL20

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:20,:]
    Y_train = Y_train[:20]

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_shuffle_fewshot20_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_shuffle_fewshot20_2.csv")

ICL500

In [ ]:
from pathlib import Path
#DATA_DIR = Path("Tabpfn_data/linear_exp_all_positive2")


rows = []
files = sorted(DATA_DIR.glob("dataset_*.csv"))[:100]
for csv_path in files:
#for csv_path in sorted(DATA_DIR.glob("dataset_*.csv")):
    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Y"])
    Y = df["Y"]
    X_train, X_test, Y_train, Y_test = split_data(X, Y)
    X_train = X_train.iloc[:500,:]
    Y_train = Y_train[:500]

    model = TabPFNRegressor(random_state=123456)
    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    MAE, RMSE, MAPE = performance_eval(y_pred, Y_test)

    rows.append({"csv": str(csv_path), "mae": MAE, "rmse": RMSE, "mape": MAPE})

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "results_shuffle_fewshot500_2.csv", index=False)
df[["mae", "rmse", "mape"]].describe().to_csv(DATA_DIR / "summary_statistics_shuffle_fewshot500_2.csv")